# Гибридная рекомендательная система для музыкального сервиса

**Задача:** сформировать персональные рекомендации треков на основе истории прослушиваний и эмбеддингов контента.

**Стек:** Hugging Face Datasets (streaming), pandas, SciPy, scikit-learn, NumPy.

Реализованы два источника кандидатов: user-based collaborative filtering и content-based retrieval. Итоговый список строится ранговым объединением результатов.


## Схема решения

1. Собрать ограниченную выборку событий из Yambda без загрузки полного корпуса.
2. Рассчитать вес взаимодействия по глубине прослушивания и лайку.
3. Построить разреженную user-item матрицу.
4. Получить кандидатов от похожих пользователей и по близости профиля к эмбеддингам треков.
5. Объединить ранги и проверить, что уже прослушанные треки не попадают в выдачу.


In [ ]:
from collections import defaultdict

import numpy as np
import pandas as pd
from datasets import load_dataset
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

MAX_LISTENS = 200_000
MAX_LIKES = 50_000
MAX_EMBEDDINGS = 20_000
TARGET_USERS = 500
MIN_USER_EVENTS = 10
TOP_K_NEIGHBORS = 30


In [ ]:
def take_stream(stream, limit: int, target_users: int | None = None) -> pd.DataFrame:
    rows = []
    users = set()
    for index, item in enumerate(stream):
        if index >= limit:
            break
        rows.append(item)
        if target_users is not None and "uid" in item:
            users.add(item["uid"])
            if len(users) >= target_users:
                break
    return pd.DataFrame(rows)


listens = take_stream(
    load_dataset(
        "yandex/yambda",
        data_dir="flat/50m",
        data_files="listens.parquet",
        split="train",
        streaming=True,
    ),
    MAX_LISTENS,
    TARGET_USERS,
)
likes = take_stream(
    load_dataset(
        "yandex/yambda",
        data_dir="flat/50m",
        data_files="likes.parquet",
        split="train",
        streaming=True,
    ),
    MAX_LIKES,
)
embeddings = take_stream(
    load_dataset(
        "yandex/yambda",
        data_files="embeddings.parquet",
        split="train",
        streaming=True,
    ),
    MAX_EMBEDDINGS,
)

required = {"uid", "item_id"}
if missing := required - set(listens.columns):
    raise ValueError(f"В событиях нет колонок: {sorted(missing)}")

active_users = listens["uid"].value_counts()
active_users = active_users[active_users >= MIN_USER_EVENTS].index
listens = listens[listens["uid"].isin(active_users)].copy()

print(f"Событий: {len(listens):,}")
print(f"Пользователей: {listens['uid'].nunique():,}")
print(f"Треков: {listens['item_id'].nunique():,}")


In [ ]:
liked_pairs = set(zip(likes.get("uid", []), likes.get("item_id", [])))


def interaction_weight(row: pd.Series) -> float:
    weight = 2.5 if (row["uid"], row["item_id"]) in liked_pairs else 1.0
    played_ratio = row.get("played_ratio_pct", np.nan)
    if pd.notna(played_ratio) and played_ratio >= 80:
        weight *= 1.5
    return weight


listens["weight"] = listens.apply(interaction_weight, axis=1)
interactions = (
    listens.groupby(["uid", "item_id"], as_index=False)["weight"].sum()
)

users = interactions["uid"].unique()
items = interactions["item_id"].unique()
user_to_index = {user_id: index for index, user_id in enumerate(users)}
item_to_index = {item_id: index for index, item_id in enumerate(items)}

matrix = csr_matrix(
    (
        interactions["weight"],
        (
            interactions["uid"].map(user_to_index),
            interactions["item_id"].map(item_to_index),
        ),
    ),
    shape=(len(users), len(items)),
)
normalized_matrix = normalize(matrix, norm="l2", axis=1)
user_similarity = cosine_similarity(normalized_matrix, dense_output=True)

density = matrix.nnz / max(matrix.shape[0] * matrix.shape[1], 1)
print(f"Матрица: {matrix.shape}; плотность: {density:.4%}")


In [ ]:
def recommend_collaborative(user_id, n: int = 10) -> list:
    if user_id not in user_to_index:
        return []

    user_index = user_to_index[user_id]
    seen_items = set(matrix[user_index].indices)
    neighbors = np.argsort(user_similarity[user_index])[::-1]
    neighbors = [idx for idx in neighbors if idx != user_index][:TOP_K_NEIGHBORS]

    scores = defaultdict(float)
    normalizers = defaultdict(float)
    for neighbor in neighbors:
        similarity = user_similarity[user_index, neighbor]
        if similarity <= 0:
            continue
        for item_index, weight in zip(
            matrix[neighbor].indices, matrix[neighbor].data
        ):
            if item_index not in seen_items:
                scores[item_index] += similarity * weight
                normalizers[item_index] += similarity

    ranked = sorted(
        (
            (item_index, score / normalizers[item_index])
            for item_index, score in scores.items()
        ),
        key=lambda pair: pair[1],
        reverse=True,
    )
    return [items[item_index] for item_index, _ in ranked[:n]]


In [ ]:
embedding_column = next(
    (name for name in ["embed", "normalized_embed"] if name in embeddings.columns),
    None,
)
content_item_ids = []
content_matrix = None
content_item_to_index = {}

if embedding_column is not None and "item_id" in embeddings.columns:
    available_items = set(items)
    content = embeddings[embeddings["item_id"].isin(available_items)].drop_duplicates(
        "item_id"
    )
    if len(content) >= 2:
        content_item_ids = content["item_id"].tolist()
        content_matrix = normalize(
            np.vstack(content[embedding_column].map(np.asarray)), norm="l2", axis=1
        )
        content_item_to_index = {
            item_id: index for index, item_id in enumerate(content_item_ids)
        }


def recommend_content(user_id, n: int = 10) -> list:
    if content_matrix is None:
        return []

    history = interactions[interactions["uid"] == user_id]
    history = history[history["item_id"].isin(content_item_to_index)]
    if history.empty:
        return []

    indices = history["item_id"].map(content_item_to_index).to_numpy()
    weights = history["weight"].to_numpy(dtype=float)
    profile = np.average(content_matrix[indices], axis=0, weights=weights)
    profile /= np.linalg.norm(profile) + 1e-12

    scores = content_matrix @ profile
    seen = set(history["item_id"])
    ranked_indices = np.argsort(scores)[::-1]
    return [
        content_item_ids[index]
        for index in ranked_indices
        if content_item_ids[index] not in seen
    ][:n]


In [ ]:
def reciprocal_rank_fusion(rankings: list[list], k: int = 60) -> list:
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, item_id in enumerate(ranking, start=1):
            scores[item_id] += 1.0 / (k + rank)
    return [
        item_id
        for item_id, _ in sorted(scores.items(), key=lambda pair: pair[1], reverse=True)
    ]


def recommend_hybrid(user_id, n: int = 10) -> list:
    collaborative = recommend_collaborative(user_id, n=n * 3)
    content = recommend_content(user_id, n=n * 3)
    seen = set(interactions.loc[interactions["uid"] == user_id, "item_id"])
    combined = reciprocal_rank_fusion([collaborative, content])
    return [item_id for item_id in combined if item_id not in seen][:n]


demo_users = interactions["uid"].value_counts().head(5).index
demo = pd.DataFrame(
    {
        "user_id": demo_users,
        "recommendations": [recommend_hybrid(user_id, 10) for user_id in demo_users],
    }
)
demo


In [ ]:
recommended_items = {
    item_id
    for user_id in demo_users
    for item_id in recommend_hybrid(user_id, 20)
}
catalog_coverage = len(recommended_items) / max(len(items), 1)
print(f"Покрытие каталога на демонстрационной группе: {catalog_coverage:.2%}")


## Что улучшить перед продакшеном

Для полноценной офлайн-оценки нужен временной holdout и метрики Recall@K/NDCG@K. На большом каталоге матрицу похожести пользователей следует заменить приближённым поиском соседей, а итоговый ранг — обучаемым ранжированием с признаками свежести и популярности.
